In [1]:
Features = [
    'item_nbr',
    'family',
    'class',
    'perishable',
    'has_float_sales',
    'is_likely_closed',
    'day_of_week',
    'is_weekend',
    'is_month_start',
    'is_month_end',
    'item_mean_sales',
    'item_std_sales',
    'sales_vs_family_mean',
    'lag_1',
    'lag_3',
    'lag_5',
    'lag_7',
    'lag_14',
    'lag_30',
    'lag_60',
    'lag_140',
    'rolling_mean_3',
    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_30',
    'rolling_mean_140',
    'rolling_std_7',
    'rolling_std_14',
    'rolling_std_30',
    'rolling_max_7',
    'rolling_min_7',
    'transactions_lag_1',
    'transactions_lag_3',
    'transactions_lag_7',
    'transactions_lag_14',
    'transactions_rolling_7',
    'transactions_rolling_14',
    'transactions_rolling_30',
    'traffic_ratio',
    'oil_price_lag_1',
    'oil_price_lag_3',
    'oil_price_lag_7',
    'oil_price_rolling_mean_7',
    'oil_price_rolling_mean_30',
    'oil_price_change_7',
    'promo_lag_1',
    'promo_lag_3',
    'promo_lag_7',
    'promo_count_7',
    'promo_count_30',
    'days_since_last_promo',
    'days_since_last_holiday',
    'days_to_next_holiday',
    'Future_is_Holiday',
    'Future_is_National',
    'Future_is_Local',
    'Future_is_likely_closed',
    'Future_day_of_week',
    'Future_is_weekend',
    'Future_is_month_start',
    'Future_is_month_end',
    'Future_On_Promotion'
]
len(Features)

62

In [2]:
Target = 'target'

In [3]:
Categorical_Features = [
    'item_nbr',
    'family',
    'class'
]

In [4]:
Future_Features_Map = {
    'is_Holiday' : 'Future_is_Holiday',
    'is_National' : 'Future_is_National',
    'is_Local' : 'Future_is_Local',
    'is_likely_closed' : 'Future_is_likely_closed',
    'day_of_week' : 'Future_day_of_week',
    'is_weekend' : 'Future_is_weekend',
    'is_month_start' : 'Future_is_month_start',
    'is_month_end' : 'Future_is_month_end',
    'onpromotion' : 'Future_On_Promotion',
    'unit_sales' : 'target'
}
len(Future_Features_Map)

10

In [2]:
horizon = 28

In [6]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

transformer = FunctionTransformer(func=np.log1p, inverse_func=np.expm1)

In [ ]:
from lightgbm import LGBMRegressor
import polars as pl
from tqdm import trange

In [8]:
def NWRMSLE_Evaluator(Dataset : pl.DataFrame, horizon : int, horizon_total : int, transformer : FunctionTransformer):
    mapping = {
        0 : 1.0,
        1 : 1.5
    }
    Added_Future_Features_Dataset = Dataset.with_columns(
            [
                pl.col(oldName).shift(-(horizon)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
            ]
        ).with_columns(
            [
                pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
            ]
        ).with_columns(
            [
                pl.col('perishable').replace_strict(mapping, default=None).alias('weight'),
                pl.col('item_nbr').shift(-horizon_total).over('item_nbr').alias('delete')
            ]
        )

    Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

    rounding_mask = Added_Future_Features_Dataset.get_column('has_float_sales').to_numpy()

    X = Added_Future_Features_Dataset.select(Features).to_pandas()
    y = Added_Future_Features_Dataset.get_column(Target).to_numpy()
    weights = Added_Future_Features_Dataset.get_column('weight').to_numpy()
    
    def NWRMSLE(y_true, y_pred):
        y_pred = transformer.inverse_transform(y_pred)
        y_pred = np.clip(y_pred, a_min=0, a_max=None)
        y_pred = np.where(rounding_mask == 0, np.round(y_pred), y_pred)
        y_pred_transformed = transformer.transform(y_pred)

        loss = (weights*(y_pred_transformed - y_true)**2).sum()/weights.sum()

        return 'NWRMSLE', loss, False
    
    return NWRMSLE, X, y

In [9]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

c:\Users\Amine\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class OptunaPruningCallback:
    def __init__(self, trial: optuna.Trial, metric_name: str):
        self.trial = trial
        self.metric_name = metric_name

    def __call__(self, env):
        for _, metric_name, score, _ in env.evaluation_result_list:
            if metric_name == self.metric_name:
                self.trial.report(score, step=env.iteration)

                if self.trial.should_prune():
                    raise optuna.TrialPruned()


In [11]:
def train_trial(trial, Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator):
    parameters = {
        'num_leaves' : trial.suggest_int('num_leaves', 16, 256),
        'max_depth' : trial.suggest_int('max_depth', 3, 15),
        'min_data_in_leaf' : trial.suggest_int('min_data_in_leaf', 10, 1000),
        'min_gain_to_split' : trial.suggest_float('min_gain_to_split', 0, 15),
        'learning_rate' : trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq' : trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1' : trial.suggest_float('lambda_l1', 1e-8, 100, log=True),
        'lambda_l2' : trial.suggest_float('lambda_l2', 1e-8, 100, log=True),
        'max_bin' : trial.suggest_int('max_bin', 4000, 7000),
        'objective' : 'regression',
        'metric' : None,
        'device' : 'cpu',
        'n_jobs' : -1,
        'verbose' : -1
    }

    model = LGBMRegressor(**parameters)

    pruning_callback = OptunaPruningCallback(trial, metric_name='NWRMSLE')

    try:
        model.fit(
            Train_X,
            Train_Y,
            categorical_feature=Categorical_Features,
            eval_set=[(Val_X, Val_Y)],
            eval_metric=nwrmsle_Evaluator,
            callbacks=[
                pruning_callback
            ]
        )
    except:
        raise optuna.TrialPruned()
    
    score = model.evals_result_["valid_0"]["NWRMSLE"][-1]

    return score

def make_objective(Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator):
    def objective(trial):
        return train_trial(trial, Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator)
    return objective

In [12]:
Train_Dataset = pl.read_csv('Store39_Dataset/MasterTrain.csv')
Val_Dataset = pl.read_csv("Store39_Dataset/MasterVal.csv")

In [13]:
import os
optuna_path = f"./optuna_studies"
os.makedirs(optuna_path, exist_ok=True)

In [ ]:
import joblib

N_TRIALS = 50

for h in trange(horizon, desc='Currently Tuning Model'):
    Added_Future_Features_Dataset = Train_Dataset.with_columns(
            [
                pl.col(oldName).shift(-(h+1)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
            ]
        ).with_columns(
            [
                pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
            ]
        ).with_columns(
            pl.col('item_nbr').shift(-horizon).over('item_nbr').alias('delete')
        )

    Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

    Train_X = Added_Future_Features_Dataset.select(Features).to_pandas()
    Train_Y = Added_Future_Features_Dataset.get_column(Target).to_numpy()

    nwrmsle_Evaluator, Val_X, Val_Y = NWRMSLE_Evaluator(Val_Dataset, h, horizon, transformer)

    sampler = TPESampler(seed=42, multivariate=True)

    pruner = MedianPruner(
        n_startup_trials = 5,
        n_warmup_steps = 30
    )

    study = optuna.create_study(
        direction = 'minimize',
        sampler = sampler,
        pruner = pruner,
        study_name= f'Model{h}_tuning'
    )
 
    study.optimize(
        make_objective(Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator),
        n_trials  = N_TRIALS,
        show_progress_bar = True
    )

    print(f"\n ============== Model{h} Results ==============")
    print("Best trial:")
    print(f"Val loss : {study.best_trial.value:.6f}")
    print(f"Params :")
    for k, v in study.best_trial.params.items():
        print(f"{k:20s} : {v}")

    joblib.dump(study, optuna_path+f'/Optuna_Model{h+1}_Study.pkl')

In [3]:
import joblib

for i in range(1, horizon + 1):
    print(f'=========== Model {i} Hyper Parameters ==============')
    study = joblib.load(f"./optuna_studies/Optuna_Model{i}_Study.pkl")
    for k, v in study.best_trial.params.items():
        print(f"{k:20s} : {v}")

=========== Model 1 Hyper Parameters ==============


c:\Users\Amine\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_leaves           : 34
max_depth            : 3
min_data_in_leaf     : 625
min_gain_to_split    : 3.0234914253866574
learning_rate        : 0.1323058700644602
feature_fraction     : 0.7409588275178136
bagging_fraction     : 0.6033488629493166
bagging_freq         : 2
lambda_l1            : 0.00391781335877298
lambda_l2            : 0.19365685947258562
max_bin              : 5741
=========== Model 2 Hyper Parameters ==============
num_leaves           : 159
max_depth            : 12
min_data_in_leaf     : 136
min_gain_to_split    : 11.334468621924424
learning_rate        : 0.07798666169912068
feature_fraction     : 0.5163805502122524
bagging_fraction     : 0.9654162374188393
bagging_freq         : 5
lambda_l1            : 0.32030649916442966
lambda_l2            : 5.671814461517291e-05
max_bin              : 5400
=========== Model 3 Hyper Parameters ==============
num_leaves           : 151
max_depth            : 12
min_data_in_leaf     : 26
min_gain_to_split    : 8.195298667865135
l